# Semana 7 — Self-Attention do Zero
**Fase 2: Deep Learning e o Ecossistema Hugging Face** · Gabarito (notebook resolvido)

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/02-Phase-2-Deep-Learning/notebooks/semana_07_gabarito.ipynb)

Leitura de conceito desta semana: [Semana-07-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/02-Phase-2-Deep-Learning/docs/Semana-07-Conceitos.md)

Use este gabarito **depois** de tentar o notebook de exercícios. Compare a sua solução, não copie.

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F, math, matplotlib.pyplot as plt
torch.manual_seed(0)

## Passo 1 — Attention em um exemplo de 3 tokens
Calcule passo a passo: afinidades, escala, softmax, média ponderada.

In [ ]:
T, d = 3, 4
q = torch.randn(T, d); k = torch.randn(T, d); v = torch.randn(T, d)
afinidade = q @ k.T                      # (T, T)
pesos = F.softmax(afinidade / math.sqrt(d), dim=-1)
saida = pesos @ v                        # (T, d)
print(pesos.round(decimals=2), "\nlinhas somam 1:", pesos.sum(-1))
assert torch.allclose(saida, F.scaled_dot_product_attention(q[None], k[None], v[None])[0], atol=1e-6)

## Passo 2 — Causal mask
Crie uma mask triangular inferior e preencha as posições proibidas com `-inf` antes do softmax.

In [ ]:
mascara = torch.tril(torch.ones(T, T)).bool()
pesos_causais = F.softmax((afinidade / math.sqrt(d)).masked_fill(~mascara, float("-inf")), dim=-1)
print(pesos_causais.round(decimals=2))
assert torch.allclose(pesos_causais.triu(1), torch.zeros(T, T))

## Passo 3 — Por que escalar por √d
Com dimensão alta e sem escala, o softmax satura em um único token.

In [ ]:
for dim in (4, 64, 1024):
    a, b = torch.randn(8, dim), torch.randn(8, dim)
    sem = F.softmax(a @ b.T, -1).max(-1).values.mean().item()
    com = F.softmax(a @ b.T / math.sqrt(dim), -1).max(-1).values.mean().item()
    print(f"d={dim:5d} | maior peso médio sem escala: {sem:.2f} | com escala: {com:.2f}")

## Passo 4 — Uma attention head como `nn.Module`
Entrada `(B, T, C)`. Projeções lineares sem bias para key, query e value, cada uma com saída `head_size`.

In [ ]:
class Cabeca(nn.Module):
    def __init__(self, C, head_size, T_max):
        super().__init__()
        self.key = nn.Linear(C, head_size, bias=False)
        self.query = nn.Linear(C, head_size, bias=False)
        self.value = nn.Linear(C, head_size, bias=False)
        self.register_buffer("tril", torch.tril(torch.ones(T_max, T_max)))
    def forward(self, x):
        B, T, C = x.shape
        k, q, v = self.key(x), self.query(x), self.value(x)
        att = q @ k.transpose(-2, -1) / math.sqrt(k.size(-1))          # (B, T, T)
        att = att.masked_fill(self.tril[:T, :T] == 0, float("-inf"))
        self.pesos = F.softmax(att, dim=-1)
        return self.pesos @ v

x = torch.randn(2, 8, 32)
print(Cabeca(32, 16, 8)(x).shape)   # esperado: (2, 8, 16)

## Passo 5 — Multi-head attention
`n_heads` heads de tamanho `C // n_heads`, concatenadas e passadas por uma projeção final `Linear(C, C)`.

In [ ]:
class MultiCabeca(nn.Module):
    def __init__(self, C, n_heads, T_max):
        super().__init__()
        self.cabecas = nn.ModuleList([Cabeca(C, C // n_heads, T_max) for _ in range(n_heads)])
        self.proj = nn.Linear(C, C)
    def forward(self, x):
        return self.proj(torch.cat([h(x) for h in self.cabecas], dim=-1))

mc = MultiCabeca(32, 4, 8); print(mc(x).shape)   # esperado: (2, 8, 32)

## Passo 6 — A attention não sabe a ordem
Embaralhe os tokens de entrada de uma attention **sem mask**: a saída é a mesma, só que embaralhada. A informação de posição precisa vir de fora.

In [ ]:
perm = torch.randperm(8); xs = x[:1]
qk = F.scaled_dot_product_attention
assert torch.allclose(qk(xs, xs, xs)[:, perm], qk(xs[:, perm], xs[:, perm], xs[:, perm]), atol=1e-5)
print("Sem posição, atenção é invariante à ordem (equivariante a permutações).")

## Passo 7 — Sinusoidal positional encoding
`PE[pos, 2i] = sin(pos / 10000^(2i/d))` e `PE[pos, 2i+1] = cos(pos / 10000^(2i/d))`.

In [ ]:
def posicional(T, d):
    pos = torch.arange(T).unsqueeze(1).float()
    freq = torch.exp(-math.log(10000.0) * torch.arange(0, d, 2).float() / d)
    pe = torch.zeros(T, d)
    pe[:, 0::2] = torch.sin(pos * freq)
    pe[:, 1::2] = torch.cos(pos * freq)
    return pe

pe = posicional(64, 32)
plt.figure(figsize=(9, 3)); plt.imshow(pe.T, aspect="auto", cmap="RdBu"); plt.xlabel("posição"); plt.ylabel("dimensão"); plt.show()
sim = F.normalize(pe, dim=1) @ F.normalize(pe, dim=1).T
print("similaridade da posição 10 com 9, 11, 30, 60:", [round(sim[10, j].item(), 2) for j in (9, 11, 30, 60)])

## Passo 8 — Attention aprende a buscar
Tarefa de brinquedo: o **último token é um ponteiro**. Seu valor `p` indica a posição cujo token deve ser copiado como resposta. Para acertar, a última posição precisa "consultar" a posição `p`: é exatamente o que query e key fazem. Treine uma layer de attention e visualize os pesos.

In [ ]:
V, T, C = 12, 10, 32
def lote(B=128):
    x = torch.randint(1, V, (B, T)); p = torch.randint(0, T - 1, (B,))
    x[:, -1] = p                                        # último token = ponteiro para a posição p
    y = x[torch.arange(B), p]                           # alvo: o token que está na posição p
    return x, y

class Mini(nn.Module):
    def __init__(self):
        super().__init__()
        self.tok = nn.Embedding(V, C); self.pos = nn.Embedding(T, C)
        self.att = MultiCabeca(C, 2, T); self.saida = nn.Linear(C, V)
    def forward(self, x):
        h = self.tok(x) + self.pos(torch.arange(x.size(1)))
        return self.saida(h + self.att(h))[:, -1]       # prevê a partir da última posição

mini = Mini(); opt = torch.optim.AdamW(mini.parameters(), lr=3e-3)
for passo in range(1500):
    xb, yb = lote()
    loss = F.cross_entropy(mini(xb), yb)
    opt.zero_grad(); loss.backward(); opt.step()
xb, yb = lote(1000); print("acurácia:", (mini(xb).argmax(-1) == yb).float().mean().item(), "| perda:", round(loss.item(), 3))

In [ ]:
xb, _ = lote(1); mini(xb)
fig, ax = plt.subplots(1, 2, figsize=(9, 4))
for a, h in zip(ax, mini.att.cabecas):
    a.imshow(h.pesos[0].detach(), cmap="viridis"); a.set_xticks(range(T), xb[0].tolist()); a.set_yticks(range(T), xb[0].tolist()); a.set_xlabel("olhando para")
plt.suptitle("pesos de atenção (última linha = posição que faz a previsão)"); plt.show()

**Reflexão:** na última linha de cada mapa, a attention está concentrada na posição indicada pelo último token? Salve a figura em `docs/` com um comentário.